In [6]:
import numpy as np
import scipy
import matplotlib.pyplot as plt


from metastable.state import FixedPointMap
from metastable.eom import EOM, Params
from metastable.manifold_inverses import calculate_manifold_inverses
from metastable.incoming_quantum_vector import extend_to_keldysh_state



map_path = "/home/paul/Projects/keldysh/metastable/00-attempt/map.npz"
fixed_point_map = FixedPointMap.load(map_path)

In [7]:
def calculate_action(bvp_result):
    
    def integrand_func(t):
        integrand = -bvp_result.sol(t,nu=1)[0]*bvp_result.sol(t)[2]
        integrand -= bvp_result.sol(t,nu=1)[1]*bvp_result.sol(t)[3]
        return integrand

    action, action_error = scipy.integrate.quad(integrand_func, 0, bvp_result.x[-1], limit=2000, epsabs=1e-2)
    return action, action_error

In [8]:
calculate_action(fixed_point_map.path_results[360,80])


In [9]:
actions_array = np.zeros(fixed_point_map.kappa_linspace.shape)

from tqdm import tqdm

for idx in tqdm(range(40, 236)):
    bvp_result = fixed_point_map.path_results[360,idx]
    action, action_error = calculate_action(bvp_result)
    actions_array[idx] = action

In [10]:
plt.plot(fixed_point_map.kappa_linspace, actions_array)
plt.xlabel(r'$\kappa$')
plt.ylabel(r'action, bright to saddle point')

In [11]:
kappa_idx = 35
res = fixed_point_map.path_results[360,kappa_idx]
fig, axes = plt.subplots(1,1,figsize=(8, 8))
t_plot = np.linspace(0, res.x[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()

action, action_error = calculate_action(res)

print(action)
print(fixed_point_map.kappa_linspace[kappa_idx])

In [12]:
fixed_point_map.fixed_points[374,0]

In [26]:
kappa_idx = 50


fig, axes = plt.subplots(1, 1, figsize=(8, 8))

axes.plot(fixed_point_map.kappa_linspace, actions_array)
axes.set_xlabel(r'$\kappa$')
axes.set_ylabel(r'action, bright to saddle point')

axes.axvline(fixed_point_map.kappa_linspace[kappa_idx], color='k', linestyle='--')

res = fixed_point_map.path_results[360,kappa_idx]

fig, axes = plt.subplots(1,1,figsize=(8, 8))
t_plot = np.linspace(0, res.x[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()

action, action_error = calculate_action(res)

print(action)
print(fixed_point_map.kappa_linspace[kappa_idx])

In [36]:
fixed_point_map.fixed_points[360, kappa_idx]

In [35]:
fig, axes = plt.subplots(1,1,figsize=(8, 8))

t_plot = np.linspace(0, res.x[-1], 1001)
axes.plot(t_plot, res.sol(t_plot)[0])


In [57]:
from metastable.generate_boundary_conditions import generate_boundary_condition_func
from metastable.generate_guess import generate_guess_from_sol

seed_epsilon_idx = 360
seed_kappa_idx = 81
res = fixed_point_map.path_results[seed_epsilon_idx, seed_kappa_idx]


epsilon_idx = seed_epsilon_idx


params = Params(
    epsilon=fixed_point_map.epsilon_linspace[epsilon_idx],
    kappa=fixed_point_map.kappa_linspace[kappa_idx],
    delta=fixed_point_map.delta,
    chi=fixed_point_map.chi,
)
eom = EOM(params=params)
classical_saddle_point = fixed_point_map.fixed_points[epsilon_idx, kappa_idx, 0]
classical_focus_point = fixed_point_map.fixed_points[epsilon_idx, kappa_idx, 1]
print(params)

keldysh_saddle_point = extend_to_keldysh_state(classical_saddle_point)
keldysh_focus_point = extend_to_keldysh_state(classical_focus_point)


boundary_condition_func = generate_boundary_condition_func(keldysh_saddle_point, keldysh_focus_point, params)


t_guess, y_guess = generate_guess_from_sol(bvp_result=res, t_end=8.0)
wrapper = lambda x, y: eom.y_dot_func(y)
res = scipy.integrate.solve_bvp(
    wrapper, boundary_condition_func, t_guess, y_guess, tol=1e-3, max_nodes=1000000, verbose=2
)


fig, axes = plt.subplots(1,1,figsize=(6, 6))
t_plot = np.linspace(0, t_guess[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()

print(res)



fig, axes = plt.subplots(1,1,figsize=(6, 6))

t_plot = np.linspace(0, res.x[-1], 1001)
axes.plot(t_plot, res.sol(t_plot)[0])

In [50]:
y_guess[:,:100].shape

In [51]:
extend_to_keldysh_state(fixed_point_map.fixed_points[epsilon_idx, kappa_idx, 1])

In [33]:
action, action_error = calculate_action(res)
print(action)

In [34]:
action, action_error = calculate_action(fixed_point_map.path_results[seed_epsilon_idx, seed_kappa_idx])
print(action)